# Extract Model IDs where Latest MDD = MRM Attachments
Scans all Excel files in a folder and collects Model IDs where the `Latest MDD found in` column is `MRM Attachments`.

In [ ]:
import pandas as pd
import glob
import os
import warnings
warnings.filterwarnings('ignore')

# ── CONFIG ──────────────────────────────────────────────────────────────────
FOLDER_PATH   = r'C:\Your\Folder\Path'   # <-- change to your folder
SHEET_NAME    = 'General Model & Review Summary'  # sheet name from your files

# Row/column locations (0-indexed) based on your spreadsheet layout:
# Row 9  (index 9)  = the data row with Model ID, Latest MDD, etc.
# Adjust HEADER_ROW if your files have a different header row
HEADER_ROW    = 8   # 0-indexed row that acts as column headers (row 9 in Excel = index 8)
MODEL_ID_COL  = 'Model ID'               # column B header text
MDD_COL       = 'Latest MDD found in\nDevelopment Tasks or MRM\nAttachments Section or\nOther sections?'  # column C header
MRM_VALUE     = 'MRM Attachments'        # the value to filter on
# ────────────────────────────────────────────────────────────────────────────

excel_files = glob.glob(os.path.join(FOLDER_PATH, '*.xlsx')) + \
              glob.glob(os.path.join(FOLDER_PATH, '*.xlsm'))

print(f'Found {len(excel_files)} Excel file(s) in: {FOLDER_PATH}\n')

matched_model_ids = []
errors            = []

for filepath in excel_files:
    filename = os.path.basename(filepath)
    try:
        # Read all sheets to handle slight header-row variations across files
        df = pd.read_excel(
            filepath,
            sheet_name=SHEET_NAME,
            header=None,          # read raw so we can locate headers ourselves
            dtype=str
        )

        # ── Locate the header row dynamically ────────────────────────────
        # Look for the row that contains 'Model ID' anywhere in it
        header_idx = None
        for i, row in df.iterrows():
            if row.astype(str).str.contains('Model ID', case=False, na=False).any():
                header_idx = i
                break

        if header_idx is None:
            errors.append((filename, 'Could not find "Model ID" header row'))
            continue

        # Re-read with the detected header row
        df = pd.read_excel(
            filepath,
            sheet_name=SHEET_NAME,
            header=header_idx,
            dtype=str
        )
        df.columns = df.columns.str.strip()  # strip whitespace from column names

        # ── Find the 'Latest MDD' column (partial match) ─────────────────
        mdd_col_match = [c for c in df.columns if 'MRM' in str(c) or 'Latest MDD' in str(c)]
        model_id_col_match = [c for c in df.columns if 'Model ID' in str(c)]

        if not mdd_col_match or not model_id_col_match:
            errors.append((filename, f'Missing columns. Found: {list(df.columns[:8])}'))
            continue

        mdd_col      = mdd_col_match[0]
        model_id_col = model_id_col_match[0]

        # ── Filter rows where MDD column == 'MRM Attachments' ─────────────
        mask = df[mdd_col].astype(str).str.strip().str.lower() == 'mrm attachments'
        matched = df.loc[mask, model_id_col].dropna().tolist()

        for mid in matched:
            mid = str(mid).strip()
            if mid and mid.lower() != 'nan':
                matched_model_ids.append({'Model ID': mid, 'Source File': filename})

    except Exception as e:
        errors.append((filename, str(e)))

# ── Results ──────────────────────────────────────────────────────────────────
results_df = pd.DataFrame(matched_model_ids)

print(f'=== Model IDs where Latest MDD = "MRM Attachments" ===')
if results_df.empty:
    print('No matching records found.')
else:
    print(results_df.to_string(index=False))
    print(f'\nTotal: {len(results_df)} record(s) across {results_df["Source File"].nunique()} file(s)')

if errors:
    print('\n=== Files with Errors ===')
    for fname, err in errors:
        print(f'  {fname}: {err}')

In [ ]:
# ── Optional: Export results to Excel ────────────────────────────────────────
if not results_df.empty:
    output_path = os.path.join(FOLDER_PATH, 'MRM_Attachment_ModelIDs.xlsx')
    results_df.to_excel(output_path, index=False)
    print(f'Saved to: {output_path}')

# Just the Model IDs as a plain list
model_id_list = results_df['Model ID'].tolist() if not results_df.empty else []
print('\nModel IDs (list):', model_id_list)